# Getting Started

## Printing the best bid and the best ask
## 打印最优买入价和最优卖出价

In [1]:
from numba import njit

import numpy as np

# numba.njit is strongly recommended for fast backtesting.
@njit
def print_bbo(hbt):
    # Iterating until hftbacktest reaches the end of data.
    # Elapses 60-sec every iteration.
    # Time unit is the same as data's timestamp's unit.
    # Timestamp of the sample data is in nanoseconds.
    while hbt.elapse(60 * 1e9) == 0:        
        # Gets the market depth for the first asset.
        depth = hbt.depth(0)

        # Prints the best bid and the best offer.
        print(
            'current_timestamp:', hbt.current_timestamp,
            ', best_bid:', np.round(depth.best_bid, 1),
            ', best_ask:', np.round(depth.best_ask, 1)
        )
    return True

In [2]:
from hftbacktest import BacktestAsset, HashMapMarketDepthBacktest

asset = (
    BacktestAsset()
        # Sets the data to feed for this asset.
        #
        # Due to the vast size of tick-by-tick market depth and trade data,
        # loading the entire dataset into memory can be challenging,
        # particularly when backtesting across multiple days.
        # HftBacktest offers lazy loading support and is compatible with npy and preferably npz.
        #
        # For details on the normalized feed data, refer to the following documents.
        # * https://hftbacktest.readthedocs.io/en/latest/data.html    
        # * https://hftbacktest.readthedocs.io/en/latest/tutorials/Data%20Preparation.html
        .data(['usdm/ethusdc_20260302.npz'])
        # Sets the initial snapshot (optional).
        .initial_snapshot('usdm/ethusdc_20260301_eod.npz')
        # Asset type:
        # * Linear
        # * Inverse.
        # 1.0 represents the contract size, which is the value of the asset per quoted price.
        .linear_asset(1.0) 
        # HftBacktest provides two built-in latency models.
        # * constant_latency
        # * intp_order_latency
        # To implement your own latency model, please use Rust.
        # 
        # Time unit is the same as data's timestamp's unit. Timestamp of the sample data is in nanoseconds.
        # Sets the order entry latency and response latency to 10ms.
        .constant_latency(10_000_000, 10_000_000)
        # HftBacktest provides several types of built-in queue position models.
        # Please find the details in the documents below.
        # https://hftbacktest.readthedocs.io/en/latest/tutorials/Probability%20Queue%20Models.html
        #
        # To implement your own queue position model, please use Rust.
        .risk_adverse_queue_model() 
        # HftBacktest provides two built-in exchange models.
        # * no_partial_fill_exchange
        # * partial_fill_exchange
        # To implement your own exchange model, please use Rust.
        .no_partial_fill_exchange()
        # HftBacktest provides several built-in fee models.
        # * trading_value_fee_model
        # * trading_qty_fee_model
        # * flat_per_trade_fee_model
        #
        # 0.02% maker fee and 0.07% taker fee. If the fee is negative, it represents a rebate.
        # For example, -0.00005 represents a 0.005% rebate for the maker order.
        .trading_value_fee_model(0.0002, 0.0007)
        # Tick size of this asset: minimum price increasement
        .tick_size(0.1)
        # Lot size of this asset: minimum trading unit.
        .lot_size(0.001)
        # Sets the capacity of the vector that stores trades occurring in the market.
        # If you set the size, you need call `clear_last_trades` to clear the vector.
        # A value of 0 indicates that no market trades are stored. (Default)
        .last_trades_capacity(0)
)

# HftBacktest provides several types of built-in market depth implementations.
# HashMapMarketDepthBacktest constructs a Backtest using a HashMap-based market depth implementation.
# Another useful implementation is ROIVectorMarketDepth, which is utilized in ROIVectorMarketDepthBacktest.
# Please find the details in the document below.
hbt = HashMapMarketDepthBacktest([asset])

/var/folders/cq/t7280jl920z2pzmhdmv0mzw80000gn/T/ipykernel_10164/3473501741.py:30: DeprecationWarning: constant_latency() is deprecated; use constant_order_latency().
  .constant_latency(10_000_000, 10_000_000)


You can see the best bid and best ask every 60 seconds. Since the price is a 32-bit float, there may be floating-point errors. Be careful when using it. In the example, for readability, the price is rounded based on the tick size.

每 60 秒您都能看到最优买入价和最优卖出价。由于价格是一个 32 位浮点数，可能会存在浮点误差。使用时请务必小心。在示例中，为了便于阅读，价格是根据最小变动单位进行四舍五入处理的。

In [3]:
print_bbo(hbt)

current_timestamp: 1772409659992000000 , best_bid: 1937.1 , best_ask: 1937.2
current_timestamp: 1772409719992000000 , best_bid: 1937.9 , best_ask: 1938.2
current_timestamp: 1772409779992000000 , best_bid: 1939.9 , best_ask: 1940.3
current_timestamp: 1772409839992000000 , best_bid: 1939.3 , best_ask: 1939.4
current_timestamp: 1772409899992000000 , best_bid: 1940.5 , best_ask: 1940.8
current_timestamp: 1772409959992000000 , best_bid: 1943.4 , best_ask: 1943.7
current_timestamp: 1772410019992000000 , best_bid: 1939.0 , best_ask: 1939.2
current_timestamp: 1772410079992000000 , best_bid: 1944.2 , best_ask: 1944.3
current_timestamp: 1772410139992000000 , best_bid: 1944.4 , best_ask: 1944.5
current_timestamp: 1772410199992000000 , best_bid: 1945.9 , best_ask: 1946.0
current_timestamp: 1772410259992000000 , best_bid: 1948.4 , best_ask: 1948.5
current_timestamp: 1772410319992000000 , best_bid: 1946.2 , best_ask: 1946.5
current_timestamp: 1772410379992000000 , best_bid: 1945.2 , best_ask: 1945.4

True

HftBacktest cannot be reused. Therefore, after using the backtest, make sure to close it. If you use the backtest after closing, it will crash.

HftBacktest 无法重复使用。因此，在使用完回测功能后，请务必将其关闭。如果在关闭后继续使用回测，将会导致程序崩溃。

In [4]:
_ = hbt.close()

## Feeding the data

## 输入数据

When you possess adequate memory, preloading the data into memory and providing it as input will be more efficient than lazy-loading during repeated backtesting.

当您具备足够的内存时，将数据预先加载到内存中并作为输入提供，要比在反复的回测过程中进行延迟加载更加高效。

In [5]:
btcusdt_20230809 = np.load('usdm/ethusdc_20260302.npz')['data']
btcusdt_20230808_eod = np.load('usdm/ethusdc_20260301_eod.npz')['data']

asset = (
    BacktestAsset()
        .data([btcusdt_20230809])
        .initial_snapshot(btcusdt_20230808_eod)
        .linear_asset(1.0) 
        .constant_latency(10_000_000, 10_000_000)
        .risk_adverse_queue_model() 
        .no_partial_fill_exchange()
        .trading_value_fee_model(0.0002, 0.0007)
        .tick_size(0.1)
        .lot_size(0.001)
)

/var/folders/cq/t7280jl920z2pzmhdmv0mzw80000gn/T/ipykernel_10164/1324934182.py:9: DeprecationWarning: constant_latency() is deprecated; use constant_order_latency().
  .constant_latency(10_000_000, 10_000_000)


In [6]:
hbt = HashMapMarketDepthBacktest([asset])

print_bbo(hbt)

_ = hbt.close()

current_timestamp: 1772409659992000000 , best_bid: 1937.1 , best_ask: 1937.2
current_timestamp: 1772409719992000000 , best_bid: 1937.9 , best_ask: 1938.2
current_timestamp: 1772409779992000000 , best_bid: 1939.9 , best_ask: 1940.3
current_timestamp: 1772409839992000000 , best_bid: 1939.3 , best_ask: 1939.4
current_timestamp: 1772409899992000000 , best_bid: 1940.5 , best_ask: 1940.8
current_timestamp: 1772409959992000000 , best_bid: 1943.4 , best_ask: 1943.7
current_timestamp: 1772410019992000000 , best_bid: 1939.0 , best_ask: 1939.2
current_timestamp: 1772410079992000000 , best_bid: 1944.2 , best_ask: 1944.3
current_timestamp: 1772410139992000000 , best_bid: 1944.4 , best_ask: 1944.5
current_timestamp: 1772410199992000000 , best_bid: 1945.9 , best_ask: 1946.0
current_timestamp: 1772410259992000000 , best_bid: 1948.4 , best_ask: 1948.5
current_timestamp: 1772410319992000000 , best_bid: 1946.2 , best_ask: 1946.5
current_timestamp: 1772410379992000000 , best_bid: 1945.2 , best_ask: 1945.4

## Getting the market depth

## 获取市场深度

In [7]:
@njit
def print_3depth(hbt):
    while hbt.elapse(60 * 1e9) == 0:
        print('current_timestamp:', hbt.current_timestamp)

        # Gets the market depth for the first asset, in the same order as when you created the backtest.
        depth = hbt.depth(0)

        # a key of bid_depth or ask_depth is price in ticks.
        # (integer) price_tick = price / tick_size
        i = 0
        for tick_price in range(depth.best_ask_tick, depth.best_ask_tick + 100):
            qty = depth.ask_qty_at_tick(tick_price)
            if qty > 0:
                print(
                    'ask: ',
                    qty,
                    '@',
                    np.round(tick_price * depth.tick_size, 1)
                )
                
                i += 1
                if i == 3:
                    break
        i = 0
        for tick_price in range(depth.best_bid_tick, max(depth.best_bid_tick - 100, 0), -1):
            qty = depth.bid_qty_at_tick(tick_price)
            if qty > 0:
                print(
                    'bid: ',
                    qty,
                    '@',
                    np.round(tick_price * depth.tick_size, 1)
                )
            
                i += 1
                if i == 3:
                    break
    return True

In [8]:
hbt = HashMapMarketDepthBacktest([asset])

print_3depth(hbt)

_ = hbt.close()

current_timestamp: 1772409659992000000
ask:  0.2 @ 1937.2
ask:  4.163 @ 1937.5
ask:  2.029 @ 1937.7
bid:  0.046 @ 1937.1
bid:  0.011 @ 1937.0
bid:  0.022 @ 1936.9
current_timestamp: 1772409719992000000
ask:  4.231 @ 1938.2
ask:  7.595 @ 1938.3
ask:  0.944 @ 1938.4
bid:  2.809 @ 1937.9
bid:  0.022 @ 1937.7
bid:  1.274 @ 1937.6
current_timestamp: 1772409779992000000
ask:  2.159 @ 1940.3
ask:  0.115 @ 1940.4
ask:  1.726 @ 1940.5
bid:  0.053 @ 1939.9
bid:  6.335 @ 1939.7
bid:  0.398 @ 1939.5
current_timestamp: 1772409839992000000
ask:  3.968 @ 1939.4
ask:  3.437 @ 1939.5
ask:  0.153 @ 1939.6
bid:  0.011 @ 1939.3
bid:  0.064 @ 1939.2
bid:  0.022 @ 1939.1
current_timestamp: 1772409899992000000
ask:  3.861 @ 1940.8
ask:  4.715 @ 1940.9
ask:  1.372 @ 1941.0
bid:  0.011 @ 1940.5
bid:  0.064 @ 1940.4
bid:  0.011 @ 1940.3
current_timestamp: 1772409959992000000
ask:  0.012 @ 1943.7
ask:  0.128 @ 1943.8
ask:  4.128 @ 1943.9
bid:  0.681 @ 1943.4
bid:  0.271 @ 1943.3
bid:  0.677 @ 1943.2
current_time

## Submitting an order

## 提交订单

In [9]:
from hftbacktest import LIMIT, GTC, NONE, NEW, FILLED, CANCELED, EXPIRED

@njit
def print_orders(hbt):
    # You can access open orders and also closed orders via hbt.orders.
    # Gets the OrderDict for the first asset.
    orders = hbt.orders(0)
    
    # hbt.orders is a dictionary, but be aware that it does not support all dict methods, and its keys are order_id (int).
    order_values = orders.values()
    while order_values.has_next():
        order = order_values.get()
    
        order_status = ''
        if order.status == NONE:
            order_status = 'NONE' # Exchange hasn't received an order yet.
        elif order.status == NEW:
            order_status = 'NEW'
        elif order.status == FILLED:
            order_status = 'FILLED'
        elif order.status == CANCELED:
            order_status = 'CANCELED'
        elif order.status == EXPIRED:
            order_status = 'EXPIRED' 
            
        order_req = ''
        if order.req == NONE:
            order_req = 'NONE'
        elif order.req == NEW:
            order_req = 'NEW'
        elif order.req == CANCELED:
            order_req = 'CANCEL'
            
        print(
            'current_timestamp:', hbt.current_timestamp, 
             ', order_id:', order.order_id,
             ', order_price:', np.round(order.price, 1),
             ', order_qty:', order.qty,
             ', order_status:', order_status,
             ', order_req:', order_req
        )

@njit
def submit_order(hbt):
    is_order_submitted = False
    while hbt.elapse(30 * 1e9) == 0:
        # Prints open orders.
        print_orders(hbt)

        depth = hbt.depth(0)
        
        if not is_order_submitted:
            # Submits a buy order at 300 ticks below the best bid for the first asset.
            order_id = 1
            order_price = depth.best_bid - 300 * depth.tick_size
            order_qty = 1
            time_in_force = GTC # Good 'till cancel
            order_type = LIMIT
            hbt.submit_buy_order(0, order_id, order_price, order_qty, time_in_force, order_type, False)
            is_order_submitted = True
    return True

In [10]:
hbt = HashMapMarketDepthBacktest([asset])

submit_order(hbt)

_ = hbt.close()

current_timestamp: 1772409659992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409689992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409719992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409749992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409779992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409809992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409839992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409869992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.

## Clearing inactive orders (FILLED, CANCELED, EXPIRED)

## 清除无效订单 (FILLED, CANCELED, EXPIRED)

In [11]:
from hftbacktest import GTC

@njit
def clear_inactive_orders(hbt):
    is_order_submitted = False
    while hbt.elapse(30 * 1e9) == 0:
        print_orders(hbt)
        
        # Removes inactive(FILLED, CANCELED, EXPIRED) orders from hbt.orders for the first asset.
        hbt.clear_inactive_orders(0)

        depth = hbt.depth(0)
        
        if not is_order_submitted:
            order_id = 1
            order_price = depth.best_bid - 300 * depth.tick_size
            order_qty = 1
            time_in_force = GTC
            order_type = LIMIT
            hbt.submit_buy_order(0, order_id, order_price, order_qty, time_in_force, order_type, False)
            is_order_submitted = True
    return True

In [12]:
hbt = HashMapMarketDepthBacktest([asset])

clear_inactive_orders(hbt)

_ = hbt.close()

current_timestamp: 1772409659992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409689992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409719992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409749992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409779992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409809992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409839992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409869992000000 , order_id: 1 , order_price: 1907.2 , order_qty: 1.

## Watching a order status - pending due to order latency

## 查看订单状态 - 由于订单延迟而挂起

In [13]:
from hftbacktest import GTC

@njit
def watch_pending(hbt):
    is_order_submitted = False
    # Elapses 0.01-sec every iteration.
    while hbt.elapse(0.01 * 1e9) == 0:
        print_orders(hbt)
        
        hbt.clear_inactive_orders(0)

        depth = hbt.depth(0)
        
        if not is_order_submitted:
            order_id = 1
            order_price = depth.best_bid - 300 * depth.tick_size
            order_qty = 1
            time_in_force = GTC
            order_type = LIMIT
            hbt.submit_buy_order(0, order_id, order_price, order_qty, time_in_force, order_type, False)
            is_order_submitted = True
            
        # Prevents too many prints
        orders = hbt.orders(0)
        order = orders.get(order_id)
        if order.status == NEW:
            return False
    return True

The `order_status` is `None` until the acceptance message is received.

在收到确认消息之前，`order_status`始终为`None`。


In [14]:
hbt = HashMapMarketDepthBacktest([asset])

watch_pending(hbt)

_ = hbt.close()

current_timestamp: 1772409600012000000 , order_id: 1 , order_price: 1908.9 , order_qty: 1.0 , order_status: NONE , order_req: NEW
current_timestamp: 1772409600022000000 , order_id: 1 , order_price: 1908.9 , order_qty: 1.0 , order_status: NEW , order_req: NONE


## Waiting for an order response

## 等待订单响应

In [15]:
from hftbacktest import GTC

@njit
def wait_for_order_response(hbt):
    order_id = 0
    is_order_submitted = False
    while hbt.elapse(0.01 * 1e9) == 0:
        print_orders(hbt)
        
        hbt.clear_inactive_orders(0)
        
        # Prevents too many prints
        orders = hbt.orders(0)
        if order_id in orders:
            if orders.get(order_id).status == NEW:
                return False

        depth = hbt.depth(0)
        
        if not is_order_submitted:
            order_id = 1
            order_price = depth.best_bid
            order_qty = 1
            time_in_force = GTC
            order_type = LIMIT
            hbt.submit_buy_order(0, order_id, order_price, order_qty, time_in_force, order_type, False)
            # Waits for the order response for a given order id for the first asset.
            print('an order is submitted at', hbt.current_timestamp)

            # Timeout is set 1-second.
            hbt.wait_order_response(0, order_id, 1 * 1e9)
            print('an order response is received at', hbt.current_timestamp)
            is_order_submitted = True
    return True

Since the `ConstantLatency` model is used, the round-trip latency is exactly 200ms. Ideally, using historical order latency data collected from the live market is the best approach. However, if this data is not available, starting with artificially generated order latency based on feed latency is another option. We will explore this in the following examples.

由于采用了“恒定延迟”模型，往返延迟时间恰好为 200 毫秒。理想情况下，使用从实时市场收集的历史订单延迟数据是最优方法。然而，如果此数据不可用，基于馈送延迟人工生成的订单延迟也是一个选择。我们将在以下示例中对此进行探讨。

In [16]:
hbt = HashMapMarketDepthBacktest([asset])

wait_for_order_response(hbt)

_ = hbt.close()

an order is submitted at 1772409600002000000
an order response is received at 1772409600022000000
current_timestamp: 1772409600032000000 , order_id: 1 , order_price: 1938.9 , order_qty: 1.0 , order_status: NEW , order_req: NONE


## Printing position, balance, fee, and equity

## 打印仓位、余额、费用及资产净值

In [17]:
@njit
def position(hbt):
    is_order_submitted = False
    while hbt.elapse(60 * 1e9) == 0:
        print_orders(hbt)
        
        hbt.clear_inactive_orders(0)
        
        # Prints position
        print(
            'current_timestamp:', hbt.current_timestamp,
            ', position:', hbt.position(0),
            ', balance:', hbt.state_values(0).balance,
            ', fee:', hbt.state_values(0).fee
        )

        depth = hbt.depth(0)
        
        if not is_order_submitted:
            order_id = 1
            order_price = depth.best_bid
            order_qty = 1
            time_in_force = GTC
            order_type = LIMIT
            hbt.submit_buy_order(0, order_id, order_price, order_qty, time_in_force, order_type, False)
            
            # Timeout is set 1-second.
            hbt.wait_order_response(0, order_id, 1e9)
            is_order_submitted = True
    return True

In [18]:
hbt = HashMapMarketDepthBacktest([asset])

position(hbt)

_ = hbt.close()

current_timestamp: 1772409659992000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772409720012000000 , order_id: 1 , order_price: 1937.1 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409720012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772409780012000000 , order_id: 1 , order_price: 1937.1 , order_qty: 1.0 , order_status: FILLED , order_req: NONE
current_timestamp: 1772409780012000000 , position: 1.0 , balance: -1937.1000000000001 , fee: 0.38742000000000004
current_timestamp: 1772409840012000000 , position: 1.0 , balance: -1937.1000000000001 , fee: 0.38742000000000004
current_timestamp: 1772409900012000000 , position: 1.0 , balance: -1937.1000000000001 , fee: 0.38742000000000004
current_timestamp: 1772409960012000000 , position: 1.0 , balance: -1937.1000000000001 , fee: 0.38742000000000004
current_timestamp: 1772410020012000000 , position: 1.0 , balance: -1937.1000000000001 , fee: 0.38742000000000004
current_ti

## Canceling an open order

## 取消未成交订单

In [19]:
@njit
def submit_and_cancel_order(hbt):
    is_order_submitted = False
    while hbt.elapse(0.1 * 1e9) == 0:
        print_orders(hbt)
        
        hbt.clear_inactive_orders(0)
        
        # Cancels if there is an open order
        orders = hbt.orders(0)
        order_values = orders.values()
        while order_values.has_next():
            order = order_values.get()
            
            # an order is only cancellable if order status is NEW.
            # cancel request is negated if the order is already filled or filled before cancel request is processed.
            if order.cancellable:
                hbt.cancel(0, order.order_id, False)
                # You can see status still NEW and see req CANCEL.
                print_orders(hbt)
                # cancels request also has order entry/response latencies the same as submitting.
                hbt.wait_order_response(0, order.order_id, 1e9)
       
        if not is_order_submitted:
            depth = hbt.depth(0)
            
            order_id = 1
            order_price = depth.best_bid - 100 * depth.tick_size
            order_qty = 1
            time_in_force = GTC
            order_type = LIMIT
            hbt.submit_buy_order(0, order_id, order_price, order_qty, time_in_force, order_type, False)
            
            # Timeout is set 1-second.
            hbt.wait_order_response(0, order_id, 1e9)
            is_order_submitted = True
        else:
            if len(hbt.orders(0)) == 0:
                return False
    return True

In [20]:
hbt = HashMapMarketDepthBacktest([asset])

submit_and_cancel_order(hbt)

_ = hbt.close()

current_timestamp: 1772409600212000000 , order_id: 1 , order_price: 1928.9 , order_qty: 1.0 , order_status: NEW , order_req: NONE
current_timestamp: 1772409600212000000 , order_id: 1 , order_price: 1928.9 , order_qty: 1.0 , order_status: NEW , order_req: CANCEL
current_timestamp: 1772409600332000000 , order_id: 1 , order_price: 1928.9 , order_qty: 1.0 , order_status: CANCELED , order_req: NONE


## Market order

## 市价单

In [21]:
from hftbacktest import MARKET

@njit
def print_orders_exec_price(hbt):
    orders = hbt.orders(0)
    order_values = orders.values()
    while order_values.has_next():
        order = order_values.get()
            
        order_status = ''
        if order.status == NONE:
            order_status = 'NONE'
        elif order.status == NEW:
            order_status = 'NEW'
        elif order.status == FILLED:
            order_status = 'FILLED'
        elif order.status == CANCELED:
            order_status = 'CANCELED'
        elif order.status == EXPIRED:
            order_status = 'EXPIRED' 
            
        order_req = ''
        if order.req == NONE:
            order_req = 'NONE'
        elif order.req == NEW:
            order_req = 'NEW'
        elif order.req == CANCELED:
            order_req = 'CANCEL'
            
        print(
            'current_timestamp:', hbt.current_timestamp, 
             ', order_id:', order.order_id,
             ', order_price:', np.round(order.price, 1),
             ', order_qty:', order.qty,
             ', order_status:', order_status,
             ', exec_price:', np.round(order.exec_price, 1)
        )
        
@njit
def market_order(hbt):
    is_order_submitted = False
    while hbt.elapse(60 * 1e9) == 0:
        print_orders(hbt)
        
        hbt.clear_inactive_orders(0)

        state_values = hbt.state_values(0)
        
        print(
            'current_timestamp:', hbt.current_timestamp,
             ', position:', hbt.position(0),
             ', balance:', state_values.balance,
             ', fee:', state_values.fee
        )
        
        if not is_order_submitted:
            depth = hbt.depth(0)
            
            order_id = 1
            # Sets an arbitrary price, which does not affect MARKET orders.
            order_price = depth.best_bid
            order_qty = 1
            time_in_force = GTC
            order_type = MARKET
            hbt.submit_sell_order(0, order_id, order_price, order_qty, time_in_force, order_type, False)
            hbt.wait_order_response(0, order_id, 1e9)
            # You can see the order immediately filled.
            # Also you can see the order executed at the best bid which is different from what it was submitted at.
            print('best_bid:', depth.best_bid)
            print_orders_exec_price(hbt)            
            is_order_submitted = True
    return True

In [22]:
hbt = HashMapMarketDepthBacktest([asset])

market_order(hbt)

_ = hbt.close()

current_timestamp: 1772409659992000000 , position: 0.0 , balance: 0.0 , fee: 0.0
best_bid: 1937.1000000000001
current_timestamp: 1772409660012000000 , order_id: 1 , order_price: 1937.1 , order_qty: 1.0 , order_status: FILLED , exec_price: 1937.1
current_timestamp: 1772409720012000000 , order_id: 1 , order_price: 1937.1 , order_qty: 1.0 , order_status: FILLED , order_req: NONE
current_timestamp: 1772409720012000000 , position: -1.0 , balance: 1937.1000000000001 , fee: 1.3559700000000001
current_timestamp: 1772409780012000000 , position: -1.0 , balance: 1937.1000000000001 , fee: 1.3559700000000001
current_timestamp: 1772409840012000000 , position: -1.0 , balance: 1937.1000000000001 , fee: 1.3559700000000001
current_timestamp: 1772409900012000000 , position: -1.0 , balance: 1937.1000000000001 , fee: 1.3559700000000001
current_timestamp: 1772409960012000000 , position: -1.0 , balance: 1937.1000000000001 , fee: 1.3559700000000001
current_timestamp: 1772410020012000000 , position: -1.0 , bal

## GTX, Post-Only order

In [23]:
from hftbacktest import GTX

@njit
def submit_gtx(hbt):
    is_order_submitted = False
    while hbt.elapse(60 * 1e9) == 0:
        print_orders(hbt)
        
        hbt.clear_inactive_orders(0)
        
        state_values = hbt.state_values(0)
        
        print(
            'current_timestamp:', hbt.current_timestamp,
             ', position:', hbt.position(0),
             ', balance:', state_values.balance,
             ', fee:', state_values.fee
        )
        
        if not is_order_submitted:
            depth = hbt.depth(0)
            
            order_id = 1
            # Sets a deep price in the opposite side and it will be rejected by GTX.
            order_price = depth.best_bid - 100 * depth.tick_size
            order_qty = 1
            time_in_force = GTX
            order_type = LIMIT
            hbt.submit_sell_order(0, order_id, order_price, order_qty, time_in_force, order_type, False)
            hbt.wait_order_response(0, order_id, 1e9)
            is_order_submitted = True
    return True

In [24]:
hbt = HashMapMarketDepthBacktest([asset])

submit_gtx(hbt)

_ = hbt.close()

current_timestamp: 1772409659992000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772409720012000000 , order_id: 1 , order_price: 1927.1 , order_qty: 1.0 , order_status: EXPIRED , order_req: NONE
current_timestamp: 1772409720012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772409780012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772409840012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772409900012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772409960012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772410020012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772410080012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772410140012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772410200012000000 , position: 0.0 , balance: 0.0 , fee: 0.0
current_timestamp: 1772410260012000000 , position: 0.0 ,

## Plotting BBO

In [25]:
@njit
def plot_bbo(hbt, local_timestamp, best_bid, best_ask):
    while hbt.elapse(1 * 1e9) == 0:
        # Records data points
        local_timestamp.append(hbt.current_timestamp)

        depth = hbt.depth(0)
        
        best_bid.append(depth.best_bid)
        best_ask.append(depth.best_ask)
    return True

In [26]:
# Uses Numba list for njit.
from numba.typed import List
from numba import int64, float64

import polars as pl

local_timestamp = List.empty_list(int64, allocated=10000)
best_bid = List.empty_list(float64, allocated=10000)
best_ask = List.empty_list(float64, allocated=10000)

hbt = HashMapMarketDepthBacktest([asset])

plot_bbo(hbt, local_timestamp, best_bid, best_ask)

hbt.close()

df = pl.DataFrame({'timestamp': local_timestamp, 'best_bid': best_bid, 'best_ask': best_ask})
df = df.with_columns(
    pl.from_epoch('timestamp', time_unit='ns')
)

df.plot(x='timestamp')

TypeError: 'DataFramePlot' object is not callable

## Printing stats

## 打印统计数据



In [ ]:
@njit
def submit_order_stats(hbt, recorder):
    buy_order_id = 1
    sell_order_id = 2
    half_spread = 5 * hbt.depth(0).tick_size
    
    while hbt.elapse(1 * 1e9) == 0:
        hbt.clear_inactive_orders(0)

        depth = hbt.depth(0)
        
        mid_price = (depth.best_bid + depth.best_ask) / 2.0
        
        if buy_order_id not in hbt.orders(0):
            order_price = round((mid_price - half_spread) / depth.tick_size) * depth.tick_size
            order_qty = 1
            time_in_force = GTX
            order_type = LIMIT
            hbt.submit_buy_order(0, buy_order_id, order_price, order_qty, time_in_force, order_type, False)
        else:
            hbt.cancel(0, buy_order_id, False)
            
        if sell_order_id not in hbt.orders(0):
            order_price = round((mid_price + half_spread) / depth.tick_size) * depth.tick_size
            order_qty = 1
            time_in_force = GTX
            order_type = LIMIT
            hbt.submit_sell_order(0, sell_order_id, order_price, order_qty, time_in_force, order_type, False)
        else:
            hbt.cancel(0, sell_order_id, False)
            
        recorder.record(hbt)
    return True

In [ ]:
from hftbacktest import Recorder

hbt = HashMapMarketDepthBacktest([asset])

recorder = Recorder(
    # The number of assets
    hbt.num_assets,
    # The buffer size for records
    1000000
)

submit_order_stats(hbt, recorder.recorder)

_ = hbt.close()

You can get recorded states using the `get` method with the asset number.

您可以使用带有资产编号的`get`方法来获取记录状态。

In [ ]:
recorder.get(0)

Additionally, the `to_npz` method saves all records into an npz file, with the asset number as the key for the data.

此外，`to_npz` 方法会将所有记录保存到一个 npz 文件中，其中资产编号将作为数据的键。

In [ ]:
recorder.to_npz('example_record.npz')

HftBacktest also provides a performance reporting tool based on the records. Please see the details here.

HftBacktest 还提供了一个基于记录的性能报告工具。详情请见此处。

In [ ]:
from hftbacktest.stats import LinearAssetRecord

# Constructs the LinearAssetRecord from the recorded data.
record = LinearAssetRecord(recorder.get(0))

# Generates the statistics.
# You can generate monthly and daily statistics, as well as custom metrics.
stats = record.stats()

# Prints the summary.
stats.summary()

In [ ]:
stats.plot()

Bokeh using Holoviews is also supported.

In [ ]:
import holoviews as hv
hv.extension('bokeh')

stats.plot(backend='holoviews')